In [5]:
# Cell 1 - Project Paths

import re
import pickle
import pandas as pd
import numpy as np
import wandb
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)



RUN_NAME = "factorized_vae_20260622_185710"

RUN_DIR = (

    PROJECT_ROOT
    /
    "outputs"
    /
    "runs"
    /
    RUN_NAME
)

METRICS_DIR = (
    RUN_DIR
    /
    "metrics"
)

LATENT_DIR = (
    RUN_DIR
    /
    "latents"
)

OLD_LOG = (

    PROJECT_ROOT
    /
    "outputs"
    /
    "logs"
    /
    "training_log_1.txt"
)

TRAIN_CSV = METRICS_DIR / "train.csv"

VAL_CSV = METRICS_DIR / "val.csv"

MEMORY_CSV = METRICS_DIR / "memory.csv"

BETA_CSV = METRICS_DIR / "beta.csv"

HISTORY_PKL = METRICS_DIR / "history.pkl"

LATENT_PKL = LATENT_DIR / "latent_history.pkl"

CONSOLE_LOG = METRICS_DIR / "training_console.log"

print()

print("Run Directory")

print(RUN_DIR)

print()

print("Old Log")

print(OLD_LOG)

d:\IITG\Projects\audio_factor_disentanglement_v2

Run Directory
d:\IITG\Projects\audio_factor_disentanglement_v2\outputs\runs\factorized_vae_20260622_185710

Old Log
d:\IITG\Projects\audio_factor_disentanglement_v2\outputs\logs\training_log_1.txt


In [2]:
# Cell 2 - Parse Previous Training Log

text = OLD_LOG.read_text(
    encoding="utf-8",
    errors="ignore"
)

epoch_blocks = re.findall(

    r"Epoch\s+(\d+)/300(.*?)(?=Epoch\s+\d+/300|Run summary:)",

    text,

    flags=re.S
)

records = []

for epoch, block in epoch_blocks:

    record = {

        "epoch": int(epoch)
    }

    patterns = {

        "train_total":
        r"Train Total\s*:\s*([-\d\.]+)",

        "val_total":
        r"Val Total\s*:\s*([-\d\.]+)",

        "tc":
        r"TC\s*:\s*([-\d\.]+)",

        "disc_loss":
        r"Disc Loss\s*:\s*([-\d\.]+)",

        "disc_real_acc":
        r"Disc Real.*?:\s*([-\d\.]+)",

        "disc_perm_acc":
        r"Disc Perm.*?:\s*([-\d\.]+)",

        "lr":
        r"LR\s*:\s*([-\d\.eE]+)",

        "epoch_time":
        r"Epoch Time\s*:\s*([-\d\.]+)",

        "ram_gb":
        r"RAM\s*:\s*([-\d\.]+)"
    }

    for key, pattern in patterns.items():

        m = re.search(pattern, block)

        record[key] = (

            float(m.group(1))

            if m

            else np.nan
        )

    records.append(record)

old_history = pd.DataFrame(records)

display(old_history.head())

print()

print(

    "Recovered Epochs:",

    len(old_history)
)

,epoch,train_total,val_total,tc,disc_loss,disc_real_acc,disc_perm_acc,lr,epoch_time,ram_gb
0,1,5.615251,5.498284,0.0,0.694085,0.5153,0.4847,0.0005,1403.25,18.47
1,2,5.335074,5.539348,0.0,0.694618,0.4962,0.4962,0.0005,1317.25,15.48
2,3,5.336611,5.487402,0.0,0.694899,0.5076,0.5115,0.0005,1312.24,13.34
3,4,5.320140,5.670992,0.0,0.694535,0.4962,0.4847,0.0005,1330.94,13.17
4,5,5.327187,5.462345,0.0,0.695064,0.5115,0.4962,0.0005,1325.05,18.49



Recovered Epochs: 132


In [3]:
# Cell 3 - Merge Old Console Log

old_text = OLD_LOG.read_text(
    encoding="utf-8",
    errors="ignore"
)

new_text = CONSOLE_LOG.read_text(
    encoding="utf-8",
    errors="ignore"
)

marker = "Epoch 132/300"

if marker not in new_text:

    merged = old_text

    merged += "\n\n"

    merged += "=" * 80

    merged += "\n"

    merged += "========== RESUMED TRAINING ==========\n"

    merged += "=" * 80

    merged += "\n\n"

    merged += new_text

    CONSOLE_LOG.write_text(

        merged,

        encoding="utf-8"
    )

    print("Console log merged.")

else:

    print("Console log already contains previous run.")

Console log merged.


In [6]:
# Cell 4 - Rebuild Beta CSV

from src.trainers.beta_scheduler import BetaScheduler

beta_scheduler = BetaScheduler(
    PROJECT_ROOT
)

old_beta = []

for epoch in range(

    1,

    133

):

    beta = beta_scheduler.get_beta_dict(
        epoch
    )

    beta["epoch"] = epoch

    old_beta.append(
        beta
    )

old_beta = pd.DataFrame(
    old_beta
)

new_beta = pd.read_csv(
    BETA_CSV
)

new_beta = new_beta[
    new_beta["epoch"] >= 133
]

merged_beta = pd.concat(

    [

        old_beta,

        new_beta

    ],

    ignore_index=True
)

merged_beta.to_csv(

    BETA_CSV,

    index=False
)

print()

print(

    "beta.csv rebuilt:",

    len(merged_beta),

    "epochs"
)


beta.csv rebuilt: 300 epochs


In [7]:
# Cell 5 - Rebuild Memory CSV

new_memory = pd.read_csv(
    MEMORY_CSV
)

old_memory = old_history[

    [

        "epoch",

        "epoch_time",

        "ram_gb"

    ]

].copy()

old_memory["cpu_percent"] = np.nan

new_memory = new_memory[
    new_memory["epoch"] >= 133
]

memory = pd.concat(

    [

        old_memory,

        new_memory

    ],

    ignore_index=True
)

memory.to_csv(

    MEMORY_CSV,

    index=False
)

print()

print(

    "memory.csv rebuilt:",

    len(memory),

    "epochs"
)


memory.csv rebuilt: 300 epochs


In [10]:
import pickle

with open(HISTORY_PKL, "rb") as f:
    history = pickle.load(f)

print(history["train"][0].keys())
print()
print(history["train"][-1].keys())

dict_keys(['epoch', 'reconstruction', 'phase_enabled', 'phase_progress', 'logmel_loss', 'logmel_l1', 'logmel_mse', 'mr_mag_256_loss', 'mr_mag_256_l1', 'mr_mag_256_mse', 'mr_mag_512_loss', 'mr_mag_512_l1', 'mr_mag_512_mse', 'magnitude_loss', 'magnitude_l1', 'magnitude_mse', 'mr_mag_1024_loss', 'mr_mag_1024_l1', 'mr_mag_1024_mse', 'if_loss', 'if_l1', 'if_mse', 'modgd_loss', 'modgd_l1', 'modgd_mse', 'phase_sin_loss', 'phase_sin_l1', 'phase_sin_mse', 'phase_cos_loss', 'phase_cos_l1', 'phase_cos_mse', 'phase_derivative', 'derived_if_loss', 'derived_gd_loss', 'von_mises', 'phase_continuity', 'multires', 'content_kl', 'content_beta', 'speaker_kl', 'speaker_beta', 'environment_kl', 'environment_beta', 'excitation_kl', 'excitation_beta', 'fidelity_kl', 'fidelity_beta', 'kl_total', 'orthogonality', 'tc', 'total', 'disc_loss', 'disc_real_acc', 'disc_perm_acc'])

dict_keys(['epoch', 'reconstruction', 'phase_enabled', 'phase_progress', 'logmel_loss', 'logmel_l1', 'logmel_mse', 'mr_mag_256_loss', 'm

In [11]:
first = set(history["train"][0].keys())
last = set(history["train"][-1].keys())

print(last - first)
print(first - last)

{'discriminator_loss'}
set()


In [12]:
# Cell 6 - Rebuild Train and Val CSV from History

import pickle
import pandas as pd

with open(HISTORY_PKL, "rb") as f:
    history = pickle.load(f)

train_new = pd.DataFrame(history["train"])
val_new   = pd.DataFrame(history["val"])

train_new = train_new[
    train_new["epoch"] >= 133
]

val_new = val_new[
    val_new["epoch"] >= 133
]



old_train = pd.DataFrame({

    "epoch": old_history["epoch"],

    "total": old_history["train_total"],

    "tc": old_history["tc"],

    "disc_loss": old_history["disc_loss"],

    "disc_real_acc": old_history["disc_real_acc"],

    "disc_perm_acc": old_history["disc_perm_acc"]

})



old_val = pd.DataFrame({

    "epoch": old_history["epoch"],

    "total": old_history["val_total"]

})



for c in train_new.columns:

    if c not in old_train.columns:

        old_train[c] = pd.NA

old_train = old_train[
    train_new.columns
]



for c in val_new.columns:

    if c not in old_val.columns:

        old_val[c] = pd.NA

old_val = old_val[
    val_new.columns
]



train = pd.concat(

    [

        old_train,

        train_new

    ],

    ignore_index=True

)



val = pd.concat(

    [

        old_val,

        val_new

    ],

    ignore_index=True

)



train.to_csv(
    TRAIN_CSV,
    index=False
)

val.to_csv(
    VAL_CSV,
    index=False
)



print()

print("train.csv :", len(train))

print("val.csv   :", len(val))

print()

print("Columns")

print(train.columns.tolist())


train.csv : 300
val.csv   : 300

Columns
['epoch', 'reconstruction', 'phase_enabled', 'phase_progress', 'logmel_loss', 'logmel_l1', 'logmel_mse', 'mr_mag_256_loss', 'mr_mag_256_l1', 'mr_mag_256_mse', 'mr_mag_512_loss', 'mr_mag_512_l1', 'mr_mag_512_mse', 'magnitude_loss', 'magnitude_l1', 'magnitude_mse', 'mr_mag_1024_loss', 'mr_mag_1024_l1', 'mr_mag_1024_mse', 'if_loss', 'if_l1', 'if_mse', 'modgd_loss', 'modgd_l1', 'modgd_mse', 'phase_sin_loss', 'phase_sin_l1', 'phase_sin_mse', 'phase_cos_loss', 'phase_cos_l1', 'phase_cos_mse', 'phase_derivative', 'derived_if_loss', 'derived_gd_loss', 'von_mises', 'phase_continuity', 'multires', 'content_kl', 'content_beta', 'speaker_kl', 'speaker_beta', 'environment_kl', 'environment_beta', 'excitation_kl', 'excitation_beta', 'fidelity_kl', 'fidelity_beta', 'kl_total', 'orthogonality', 'tc', 'total', 'disc_loss', 'disc_real_acc', 'disc_perm_acc', 'discriminator_loss']


In [13]:
# Cell 7 - Merge Latent History

import pickle

with open(
    LATENT_PKL,
    "rb"
) as f:

    latent = pickle.load(f)

old_latent = [

    {

        "epoch": i

    }

    for i in range(

        1,

        133

    )

]

latent = old_latent + latent

with open(
    LATENT_PKL,
    "wb"
) as f:

    pickle.dump(
        latent,
        f
    )

print()

print(

    "latent_history.pkl rebuilt:",

    len(latent)

)


latent_history.pkl rebuilt: 300


In [14]:
# Cell 8 - Merge History.pkl

import pickle

with open(
    HISTORY_PKL,
    "rb"
) as f:

    history = pickle.load(f)



history["train"] = train.to_dict(
    "records"
)

history["val"] = val.to_dict(
    "records"
)

history["beta"] = pd.read_csv(
    BETA_CSV
).to_dict(
    "records"
)

history["memory"] = pd.read_csv(
    MEMORY_CSV
).to_dict(
    "records"
)



with open(
    HISTORY_PKL,
    "wb"
) as f:

    pickle.dump(
        history,
        f
    )

print()

print(

    "history.pkl rebuilt."

)


history.pkl rebuilt.


In [16]:
import pickle

with open(r"outputs\runs\factorized_vae_20260622_185710\latents\latent_history.pkl", "rb") as f:
    latent_history = pickle.load(f)

print(type(latent_history))
print(latent_history.keys())

<class 'list'>


AttributeError: 'list' object has no attribute 'keys'

In [17]:
print(type(latent_history))

print("Length:", len(latent_history))

print()

print(latent_history[0].keys())

print()

print(latent_history[0])

<class 'list'>
Length: 300

dict_keys(['epoch'])

{'epoch': 1}


In [18]:
print(latent_history[0]["epoch"])

print(latent_history[-1]["epoch"])

1
300


In [19]:
for item in latent_history[:5]:
    print(item["epoch"])

print("...")

for item in latent_history[-5:]:
    print(item["epoch"])

1
2
3
4
5
...
296
297
298
299
300
